#### Mini Math Personal Math Tutor - Inverse of a 2x2 Matrix
- Through an agent loop, with step-by-step todos, this agentic math tutor shows how to find the inverse of a given 2x2 matrix.

In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins with '{openai_api_key[:8]}'")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
openai = OpenAI()

In [ ]:
# Some lists!

todos = []
completed = []

In [ ]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [ ]:
get_todo_report()

In [ ]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [ ]:
todos, completed = [], []

create_todos([
    "Identify matrix elements a, b, c, d",
    "Calculate the determinant (ad - bc)",
    "Check if the matrix is invertible (determinant is not equal to 0)",
    "Swap a and d; negate b and c",
    "Divide each element by the determinant"
])

In [ ]:
mark_complete(1, "Matrix is [bold]A = [[3, 2], [1, 4]][/bold] → a=3, b=2, c=1, d=4")

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Plan the steps needed to find the inverse of a 2x2 matrix, then add them as todos",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark a step complete and record your working for that step",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the step to mark as complete',
                'title': 'Index',
                'type': 'integer'
            },
            'completion_notes': {
                'description': 'Show your full mathematical working for this step using Rich console markup. Include the formula, substituted values, and result.',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [ ]:
system_message = """
You are a patient and clear math tutor who specialises in linear algebra.
Your job is to teach a student how to find the inverse of a 2x2 matrix, step by step.

Use your todo tools to first plan every step, then work through each one in order.
For each step, use mark_complete with detailed working in Rich console markup:
  - Show the formula used
  - Substitute the actual numbers
  - State the result clearly

If the matrix is not invertible (determinant = 0), explain why and stop.
Provide your final answer in Rich console markup. Do not use code blocks.
Do not ask the user questions; work through the problem and give the full solution.
"""

user_message = """
Find the inverse of the following 2x2 matrix:

| 3  2 |
| 1  4 |
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": user_message}
]

In [ ]:
todos, completed = [], []
loop(messages)